# 03. ablation과 재현 설계

목표: 논문 표의 수치를 조건과 함께 구조화하고, 차이를 계산하며, 과도한 일반화를 막는 재현 checklist를 만든다.

In [ ]:
masking_results = {
    "multi-block": 54.2,
    "rasterized": 15.5,
    "single-block": 20.2,
    "random": 17.6,
}
best_baseline = max(
    score for name, score in masking_results.items() if name != "multi-block"
)
absolute_gain = masking_results["multi-block"] - best_baseline
print(f"가장 강한 비교 mask 대비 절대 향상: {absolute_gain:.1f} points")

target_results = {"representation": 66.9, "pixels": 40.7}
print("latent target 절대 향상:", target_results["representation"] - target_results["pixels"])

위 차이는 해당 표의 architecture, epoch, ImageNet-1% protocol 안에서만 해석한다. 서로 다른 표의 Top-1끼리 빼면 통제되지 않은 조건이 섞인다.

In [ ]:
# 논문의 효율 설명을 상대 단위로 확인한다.
mae_iteration_cost = 1.0
ijepa_iteration_cost = 1.07  # target representation 계산으로 약 7% 증가
mae_iterations = 5.0         # I-JEPA 대비 상대 iteration 수
ijepa_iterations = 1.0

mae_total = mae_iteration_cost * mae_iterations
ijepa_total = ijepa_iteration_cost * ijepa_iterations
relative_compute = ijepa_total / mae_total
print(f"단순 상대 계산량: MAE의 {relative_compute:.1%}")
print("주의: 실제 GPU-hours는 구현·batch·hardware와 통신에 따라 달라진다.")

In [ ]:
def reproduction_card():
    return {
        "source": ["paper version", "code commit SHA", "dataset version"],
        "masking": ["target scale/aspect", "target count", "context scale", "seed"],
        "optimization": ["global batch", "optimizer steps", "LR/WD schedule", "EMA schedule"],
        "systems": ["GPU model/count", "precision", "throughput", "peak memory"],
        "evaluation": ["frozen encoder", "pooling", "linear-probe protocol", "multiple seeds"],
        "diagnostics": ["loss", "feature variance", "pairwise cosine", "gradient norm"],
    }

for section, checks in reproduction_card().items():
    print(f"[{section}]")
    for check in checks:
        print(f"- [ ] {check}")

## 심화 과제

1. multi-block과 random mask를 같은 context ratio로 맞춰 공정한 toy ablation을 설계한다.
2. 출력 마스킹과 입력 마스킹의 정보 경로를 계산 그래프로 그린다.
3. 큰 모델이 local task에서 항상 개선되지 않은 결과에 대한 가설을 세운다.
4. 사전학습 loss, linear probe, fine-tuning 중 어떤 기준으로 checkpoint를 선택할지 사전에 정한다.
5. 평균과 표준편차를 보고하려면 최소 몇 seed가 필요한지 compute budget과 함께 결정한다.